# MorganTrace — Ingeniería de Características (Feature Engineering)

**Objetivo:** Preparar los datos del dataset IEEE-CIS para el entrenamiento del modelo.  
**Autor:** Jean Pierre Azabache  

## Pasos
1. Selección de las 50 variables más relevantes
2. Imputación de valores nulos (mediana)
3. Escalado de variables numéricas (StandardScaler)
4. División train/test estratificada (80/20)
5. Balanceo de clases con **SMOTE** (solo en train)
6. Guardado en `data/processed/`


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import joblib
import os

os.makedirs('data/processed', exist_ok=True)
os.makedirs('src/models', exist_ok=True)
print('✅ Librerías cargadas')

In [ ]:
# ── Carga y unión de datasets ──────────────────────────────────────────────────
df_trans = pd.read_csv('data/raw/train_transaction.csv')
df_ident = pd.read_csv('data/raw/train_identity.csv')
df = df_trans.merge(df_ident, on='TransactionID', how='left')
print(f'Dataset unificado: {df.shape}')

In [ ]:
# ── Selección de las 50 features más relevantes ────────────────────────────────
# Basado en importancia de XGBoost y análisis de correlación del EDA
FEATURES = [
    'TransactionAmt', 'TransactionDT',
    'V1', 'V2', 'V3', 'V4', 'V6', 'V7', 'V8', 'V9', 'V10',
    'V11', 'V12', 'V13', 'V14', 'V17', 'V19', 'V20',
    'V29', 'V30', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38',
    'V40', 'V44', 'V45', 'V47', 'V48', 'V49', 'V53', 'V54',
    'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63',
    'V64', 'V65', 'V66', 'V67', 'V70',
    'card1', 'card2', 'card3', 'card5',
]

features_ok = [f for f in FEATURES if f in df.columns]
X = df[features_ok]
y = df['isFraud']
print(f'Features seleccionadas: {len(features_ok)} de {len(FEATURES)}')

In [ ]:
# ── Imputación de nulos y escalado ─────────────────────────────────────────────
imputador = SimpleImputer(strategy='median')
X_imp = imputador.fit_transform(X)

escalador = StandardScaler()
X_esc = escalador.fit_transform(X_imp)

# Guardar transformadores para inferencia en producción
joblib.dump(imputador, 'src/models/imputador.pkl')
joblib.dump(escalador, 'src/models/escalador.pkl')
print('✅ Imputación y escalado completados')
print('   Transformadores guardados en src/models/')

In [ ]:
# ── División train/test estratificada ─────────────────────────────────────────
# Estratificado para mantener la proporción de fraude (~3.5%) en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X_esc, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Fraude en train: {y_train.mean()*100:.2f}% | en test: {y_test.mean()*100:.2f}%')

In [ ]:
# ── SMOTE: Balanceo sintético de clases ───────────────────────────────────────
# ⚠️ SMOTE solo se aplica al conjunto de ENTRENAMIENTO (nunca al test)
print('Aplicando SMOTE al conjunto de entrenamiento...')
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'Antes de SMOTE : {X_train.shape} | Fraude: {y_train.mean()*100:.2f}%')
print(f'Después de SMOTE: {X_train_bal.shape} | Fraude: {y_train_bal.mean()*100:.2f}%')

In [ ]:
# ── Guardar datos procesados ───────────────────────────────────────────────────
np.save('data/processed/X_train.npy', X_train_bal)
np.save('data/processed/y_train.npy', y_train_bal)
np.save('data/processed/X_test.npy', X_test)
np.save('data/processed/y_test.npy', y_test)
pd.Series(features_ok).to_csv('data/processed/features.csv', index=False)

print('✅ Datos guardados en data/processed/')
print('→ Continuar con 03_modeling.ipynb')